# What's the highest network number?

In [25]:
import io
import json

from IPython.display import display
import ipywidgets as widgets
import polars as pl

In [2]:
uploader = widgets.FileUpload(
    accept=".geojson",
    multiple=False
)

In [3]:
uploader

FileUpload(value=(), accept='.geojson', description='Upload')

In [7]:
nodes_geojson = json.load(io.BytesIO(uploader.value[0].content))

In [18]:
nodes = pl.DataFrame([n["properties"] for n in nodes_geojson["features"]])

In [27]:
nodes.get_column("status").unique()

status
str
"""Installed"""
"""Surveyed"""
"""Ready for Contact & Survey"""
"""Needs Maintenance or realignme…"
"""Ready for Install"""
"""Purgatory"""
"""Submitted Requests"""


In [36]:
nodes_installed = (
    nodes
    .filter(pl.col("status").is_in(["Installed", "Needs Maintenance or realignment"]))
    .with_columns(
        pl.col("title").str.extract(r"^NN(\d+).*", 1).alias("network_number").cast(pl.Int32),
    )
    .sort("title")
)

In [37]:
with pl.Config(tbl_rows=-1):
    display(nodes_installed)

title,link,status,network_number
str,str,str,i32
"""NN50 - BICAS""","""https://trello.com/c/YHdeVn7n/…","""Needs Maintenance or realignme…",50
"""NN51 - BICAS""","""https://trello.com/c/wkq1923u/…","""Installed""",51
"""NN53 - Abby, Ursula, Rosie""","""https://trello.com/c/mHDQSA2w/…","""Installed""",53
"""NN54 - Michael & Lovisa""","""https://trello.com/c/WBXioBkF/…","""Needs Maintenance or realignme…",54
"""NN55 - Outer Space 👽""","""https://trello.com/c/Nbpg39Og/…","""Installed""",55
"""NN56 - ???""","""https://trello.com/c/n9Z0dVJn/…","""Needs Maintenance or realignme…",56
"""NN57 - Steven Adger""","""https://trello.com/c/3wlzMDxc/…","""Needs Maintenance or realignme…",57
"""NN58 - Rachel Wedig""","""https://trello.com/c/ykptrbqd/…","""Installed""",58
"""NN61 - Irene Gibbs""","""https://trello.com/c/fBKREgpP/…","""Installed""",61


In [38]:
network_number_max = nodes_installed.get_column("network_number").max()

In [39]:
network_number_max

98